# Worm demo: what DiverScan found

This notebook shows the results of DiverScan on a ready-made analysis, so you can see what the tool produces before running it on your own data.

The recordings are trajectories of the nematode *C. elegans* under two conditions, `preexp` (pre-exposed) and `naive`. A model with five attention branches was trained to tell the two apart. The cells below show the same results as notebook 05 of DiverScan: how well the two conditions were told apart, where on the trajectories the model looked, and which movement features go with those places.

**How to run.** Sign in to a Google account and select `Runtime` -> `Run all`. Nothing is saved to your Google Drive: the notebook downloads the demo files it needs into the Colab session's temporary disk, and they are gone when the session ends.


In [ ]:
#@title ⬇️ Download the demo files into this session
#@markdown Nothing is written to your Google Drive. The files are downloaded into this Colab session's temporary disk and are deleted when the session ends.

import subprocess
import sys
import zipfile
from pathlib import Path

# DiverScan-release.zip in the shared Google Drive folder; only the parts of
# the archive listed in wanted_prefixes are extracted.
archive_id = "1HhIffR291MsNktWj61fIwHgrrUIgFx46"
wanted_prefixes = ["projects/worm_demo/"]

work_folder = Path("/content") if Path("/content").is_dir() else Path.cwd()
archive_path = work_folder / "DiverScan-release.zip"
project_folder = work_folder / "DiverScan-release" / "projects" / "worm_demo"

if not project_folder.is_dir():
    if not archive_path.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        gdown.download(id=archive_id, output=str(archive_path), quiet=False)
    with zipfile.ZipFile(archive_path) as archive:
        wanted = [m for m in archive.namelist() if m.startswith(tuple(wanted_prefixes))]
        archive.extractall(work_folder / "DiverScan-release", members=wanted)
    archive_path.unlink()  # the archive is no longer needed
print("demo files:", project_folder)


# ⚙️ Results

Scroll down to the results. The numbered cells can be run again on their own after changing their settings.


In [ ]:
#@title ⚙️ Setting
#@markdown Loads the demo project. There is nothing to fill in.

import os
import sys
from pathlib import Path
import pickle
import gzip
import random
import numpy as np
import pandas as pd

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Running in Google Colab environment.")
except ImportError:
    # Not running in Google Colab environment
    pass

# For reproductivity
seed = 0
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)

# The demo project, downloaded into this session by the cell above.
project_folder = Path(project_folder)

dataset_folder = project_folder / "dataset"
results_folder = project_folder / "results"
utils_folder = str(project_folder / "utils_demo")
epoch_visualization_folder = results_folder  # the cells below read their files from here

# How the demo model was set up: one animal per recording, classification,
# nodes not shuffled. The number of attention branches is read from the results.
model_type = "single"
task_type = "classification"
node_shuffle = False

# The code that draws the plots travels with the demo project.
if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import postprocessing as postprocessing_module


def read_settings_file(path):
    values = {}
    with open(path, "r") as f:
        for line in f:
            key, separator, value = line.rstrip("\n").partition(": ")
            if separator:
                values[key] = value
    return values


# The dataset: the unstandardized features of the test set, for the plots.
os.chdir(dataset_folder)
print(dataset_folder)
loaded_values = read_settings_file("settings_of_preprocessing.txt")
if "moving_window_size" in loaded_values:
    print("moving window size:", loaded_values["moving_window_size"], "frames")
with gzip.open("dataset_unstandardized_for_post_analysis.pkl", "rb") as file:
    post_analysis_dictionary = pickle.load(file)

# The results: predictions and attention of the model on the test set.
os.chdir(results_folder)
print(results_folder)
print("best epoch:", read_settings_file("best_epoch.txt").get("epoch", open("best_epoch.txt").readline().strip()))

with gzip.open("prediction_attention_namelist.pkl", "rb") as file:
    loaded_data = pickle.load(file)
attentions = loaded_data["attentions"]
class_name_list = loaded_data["class_name_list"]
feature_name_list = loaded_data["feature_name_list"]
edge_data_name_list = loaded_data["edge_data_name_list"]
number_of_attention_branches = attentions.shape[1]
if model_type == "multi":
    pred_masks = loaded_data["pred_masks"]

with gzip.open("dictionary_for_visualization.pkl", "rb") as file:
    dictionary = pickle.load(file)

print("conditions:", class_name_list)
print("attention branches:", number_of_attention_branches)


In [ ]:
#@title 1️⃣ Model performance
#@markdown You can check the performance of the attention branches.
from IPython.display import display


if task_type == "classification":

    print("⭐️ Macro averaged accuracy (test set) ⭐️")
    df = pd.read_excel("macro_averaged_accuracy.xlsx")
    df.rename(
        columns=lambda x: '' if isinstance(x, str) and x.startswith('Unnamed') else x,
        inplace=True
    )
    display(df)
    print("\n\n⭐️ Classification report (test set) ⭐️\n")

    classification_report_dictionary = pd.read_excel("classification_report.xlsx", sheet_name=None)
    postprocessing_module.show_classification_report(
        classification_report_dictionary,
        number_of_attention_branches,
    )
    print("\nFor details, see the document of sklearn.metrics.classification_report. https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html")

In [ ]:
#@title 2️⃣ PCA of attention timeseries
#@markdown You can check which attention branches are distinctive and which are similar to each other.

flatten_attentions = attentions.transpose(1, 0, 2, 3).reshape(attentions.shape[1], -1)  # (number_of_attention_branches, files*N*T)

postprocessing_module.create_interactive_pca_plot(
    flatten_attentions,
    "Default",
    scale=20,
)

In [ ]:
#@title 3️⃣ Visualization of attention

default_filename_list = dictionary[class_name_list[0]].keys()
_, number_of_nodes, _ = dictionary[class_name_list[0]][list(default_filename_list)[0]]["attentions"].shape

postprocessing_module.make_plots_and_gui(
    dictionary,
    model_type,
    class_name_list,
    default_filename_list,
    number_of_attention_branches,
    feature_name_list,
    number_of_nodes,
    dpi=100,
    xlim=None,  # e.g., (0, 100) or None
    ylim=None,  # e.g., (0, 100) or None
)

In [ ]:
#@title 4️⃣ Interactive plot
#@markdown Select the left or right plot of 3️⃣ which you want to create a interactive one.

postprocessing_module.make_interactive_plot_and_gui(
    dictionary,
    model_type,
    sampling_interval=10,
)

In [ ]:
#@title 5️⃣ Correlation and Hellinger distance
#@markdown You can see potential features that contribute to attention maps.
#@markdown - Select what to show from the dropdown; the content updates right away.
#@markdown - "Highlighted segments" are segments where the attention value exceeds the reciprocal of (the time length)(the number of nodes). 1/TN.
#@markdown - Hellinger distance is a measure of the similarity between two distributions.
#@markdown - When the Hellinger distance is 0, the two distributions are identical, and when it is 1, they are completely different.
#@markdown
#@markdown **[Multi mode] indicators** (available when model_type == "multi"):
#@markdown - `differences of basic features between nodes`: sum of absolute differences between each node's feature and those of the other nodes. Higher when the node behaves differently from the rest of the group.
#@markdown - `edge features`: mean / variance over all edges (broadcast to every node as the same value). Reflects the overall cohesion and variability of the group.
#@markdown - `incident edge features per node`: per-node sum / min / max / variance of the edges incident to the node. Distinguishes isolated individuals from central individuals on a node-by-node basis.
#@markdown - `MST features`: degree / is_leaf / incident sum / incident max on the Minimum Spanning Tree built from the distance edge. Captures structural roles such as leaf, hub, or bridge.
#@markdown - `geometric features`: distance to the group centroid / local density / head-direction alignment computed from (x, y) coordinates (only when coordinates are available).

show_options = ["Correlation (basic features)", "Correlation in highlighted segments (basic features)", "Hellinger distance in highlighted segments (basic features)", "[Multi mode] Correlation (differences of basic features between nodes)", "[Multi mode] Correlation in highlighted segments (differences of basic features between nodes)", "[Multi mode] Hellinger distance in highlighted segments (differences of basic features between nodes)", "[Multi mode] Correlation (edge features)", "[Multi mode] Correlation in highlighted segments (edge features)", "[Multi mode] Hellinger distance in highlighted segments (edge features)", "[Multi mode] Correlation (incident edge features per node)", "[Multi mode] Correlation in highlighted segments (incident edge features per node)", "[Multi mode] Hellinger distance in highlighted segments (incident edge features per node)", "[Multi mode] Correlation (MST features)", "[Multi mode] Correlation in highlighted segments (MST features)", "[Multi mode] Hellinger distance in highlighted segments (MST features)", "[Multi mode] Correlation (geometric features)", "[Multi mode] Correlation in highlighted segments (geometric features)", "[Multi mode] Hellinger distance in highlighted segments (geometric features)"]


postprocessing_module.show_selected_result_gui(
    show_options,
    "Correlation (basic features)",
    class_name_list,
    number_of_attention_branches=number_of_attention_branches,
    model_type=model_type,
)


In [ ]:
#@title 6️⃣ Histogram of features in the highlighted segments
#@markdown You can see distribution differences of feature values of two conditions in the highlighted segments.
#@markdown - Select what to show from the dropdown; the content updates right away.
#@markdown - "Highlighted segments" are segments where the attention value exceeds the reciprocal of (the time length)(the number of nodes). 1/TN.
#@markdown
#@markdown **[Multi mode] indicators** (available when model_type == "multi"):
#@markdown - `differences of basic features between nodes`: sum of absolute differences between each node's feature and those of the other nodes. Higher when the node behaves differently from the rest of the group.
#@markdown - `edge features`: mean / variance over all edges (broadcast to every node as the same value). Reflects the overall cohesion and variability of the group.
#@markdown - `incident edge features per node`: per-node sum / min / max / variance of the edges incident to the node. Distinguishes isolated individuals from central individuals on a node-by-node basis.
#@markdown - `MST features`: degree / is_leaf / incident sum / incident max on the Minimum Spanning Tree built from the distance edge. Captures structural roles such as leaf, hub, or bridge.
#@markdown - `geometric features`: distance to the group centroid / local density / head-direction alignment computed from (x, y) coordinates (only when coordinates are available).

show_options = ["Histogram in highlighted segments (basic features)", "[Multi mode] Histogram in highlighted segments (differences of basic features between nodes)", "[Multi mode] Histogram in highlighted segments (edge features)", "[Multi mode] Histogram in highlighted segments (incident edge features per node)", "[Multi mode] Histogram in highlighted segments (MST features)", "[Multi mode] Histogram in highlighted segments (geometric features)"]


postprocessing_module.show_selected_result_gui(
    show_options,
    "Histogram in highlighted segments (basic features)",
    class_name_list,
    feature_name_list=feature_name_list,
    dpi=100,
    model_type=model_type,
)


In [ ]:
#@title Model prediction of all files in the test set
#@markdown "True" indicates that the model prediction is correct, while "False" indicates it is incorrect. For more details, go to the `epochXX_for_visualization` folder.

prediction_true_false = pd.read_excel("prediction_true_false.xlsx", sheet_name=None)

postprocessing_module.show_table_and_gui(
    prediction_true_false,
    list(prediction_true_false.keys()),
    dropdown_name="condition",
)

In [ ]:
#@title Make summary tables (SummaryTable.xlsx)

import subprocess
from pathlib import Path

SCRIPT_PATH  = Path(utils_folder) / "make_summary_table.py"
INPUT_DIR    = Path(epoch_visualization_folder)
OUTPUT_XLSX  = INPUT_DIR / "SummaryTable.xlsx"

CLASS_NAMES = class_name_list
NUM_ATTENTIONS = number_of_attention_branches
MODEL_TYPE = model_type
# --------------------------------------

cmd = [
    "python",
    str(SCRIPT_PATH),
    str(INPUT_DIR),
    "--class-names", str(CLASS_NAMES),
    "--num-attentions", str(NUM_ATTENTIONS),
    "--model-type", str(MODEL_TYPE),
    "-o", str(OUTPUT_XLSX),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Saved:", OUTPUT_XLSX.resolve())


# 🤖 LLM summary

Everything above is the raw material (correlation tables, Hellinger distances, highlighted-segment histograms). The cell below turns that into a short written interpretation per attention branch.

In [ ]:
#@title 7️⃣ LLM summary
#@markdown Give this cell an **Anthropic API key** to have Claude read the correlation / Hellinger-distance / distribution tables and write an interpretation of each attention branch. The key can go in `anthropic_api_key` below or, safer, in a Colab secret named `ANTHROPIC_API_KEY` (key icon in the left sidebar).
#@markdown
#@markdown Leave both empty to instead display the report that was prepared ahead of time and bundled with this project; no key and no network call are needed for that.

anthropic_api_key = "" #@param {type:"string"}
#@markdown ---
#@markdown One phrase describing what was recorded, inserted into the prompt. Only used when an API key is given.
experimental_setting = "C. elegans tracked while exposed to an aversive odorant: `preexp` worms were pre-exposed (conditioned) to the odorant beforehand, `naive` worms were not. The user-specified location, from which `Distance from userdefined location` and `Angle from userdefined location` are measured, is the center of the arena, near the odor source." #@param {type:"string"}
#@markdown ---
model_id = "claude-opus-5-5" #@param ["claude-opus-5-5", "claude-sonnet-5-5", "claude-haiku-4-5"] {"allow-input":true}

import gzip
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display


def build_feature_comparison_table(results_folder, class_name_list, number_of_attention_branches):
    """One compact row per (branch, feature): correlation with the attention trace
    (whole trajectory and highlighted-segments-only), Hellinger distance between
    conditions, and the condition-wise mean/median/std inside the highlighted
    segments. Built from the xlsx/pkl files 5️⃣ and 6️⃣ already produced above —
    not from the raw per-frame data, which is far too large to hand to an LLM
    (thousands of values per feature per branch per condition)."""
    results_folder = Path(results_folder)
    corr_whole = pd.ExcelFile(results_folder / "correlation.xlsx")
    corr_hl = pd.ExcelFile(results_folder / "correlation_in_highlighted_segments.xlsx")
    hellinger = pd.read_excel(results_folder / "hellinger_distance.xlsx").set_index("Feature")
    with gzip.open(results_folder / "histogram.pkl", "rb") as file:
        histogram = pickle.load(file)

    corr_whole_by_condition = {c: corr_whole.parse(c, index_col=0) for c in class_name_list}
    corr_hl_by_condition = {c: corr_hl.parse(c, index_col=0) for c in class_name_list}
    cond_a, cond_b = class_name_list[0], class_name_list[1]

    rows = []
    for branch_idx in range(number_of_attention_branches):
        branch = f"Attention{branch_idx + 1}"
        key = f"{branch} Condition 1 - 2"
        if key not in histogram:
            continue
        for feature, by_condition in histogram[key].items():
            values_a = np.asarray(by_condition[0])
            values_b = np.asarray(by_condition[1])
            row = {"branch": branch, "feature": feature}
            for cond, corr_table in [(cond_a, corr_whole_by_condition[cond_a]), (cond_b, corr_whole_by_condition[cond_b])]:
                row[f"corr_whole_{cond}"] = corr_table.loc[feature, branch] if feature in corr_table.index else np.nan
            for cond, corr_table in [(cond_a, corr_hl_by_condition[cond_a]), (cond_b, corr_hl_by_condition[cond_b])]:
                row[f"corr_highlighted_{cond}"] = corr_table.loc[feature, branch] if feature in corr_table.index else np.nan
            row["hellinger_distance"] = hellinger.loc[feature, key] if feature in hellinger.index else np.nan
            row[f"mean_{cond_a}"] = float(np.mean(values_a)) if values_a.size else np.nan
            row[f"mean_{cond_b}"] = float(np.mean(values_b)) if values_b.size else np.nan
            row[f"median_{cond_a}"] = float(np.median(values_a)) if values_a.size else np.nan
            row[f"median_{cond_b}"] = float(np.median(values_b)) if values_b.size else np.nan
            row[f"std_{cond_a}"] = float(np.std(values_a)) if values_a.size else np.nan
            row[f"std_{cond_b}"] = float(np.std(values_b)) if values_b.size else np.nan
            rows.append(row)
    return pd.DataFrame(rows).round(4)


def build_timing_table(dictionary, class_name_list, number_of_attention_branches):
    """For each branch and condition: how often attention is 'highlighted' (exceeds
    1/(N*T)), and where in the trajectory those highlighted moments are
    concentrated (an attention-weighted time centroid, normalized to 0-1). Only
    correctly-classified files are used, matching how hellinger_distance.xlsx /
    histogram.pkl were built in 5️⃣ / 6️⃣."""
    rows = []
    for branch_idx in range(number_of_attention_branches):
        for condition in class_name_list:
            fractions, centroids = [], []
            for entry in dictionary[condition].values():
                if entry["prediction"][branch_idx] != entry["label"][branch_idx]:
                    continue
                attention = entry["attentions"][branch_idx, 0, :]
                num_timestamps = attention.shape[0]
                highlighted = attention > (1.0 / num_timestamps)
                fractions.append(highlighted.mean())
                if highlighted.sum() > 0:
                    normalized_time = np.arange(num_timestamps) / (num_timestamps - 1)
                    centroids.append(np.average(normalized_time[highlighted], weights=attention[highlighted]))
            rows.append({
                "branch": f"Attention{branch_idx + 1}",
                "condition": condition,
                "highlighted_timepoints_pct": round(float(np.mean(fractions)) * 100, 1) if fractions else np.nan,
                "highlighted_time_centroid_0to1": round(float(np.mean(centroids)), 3) if centroids else np.nan,
                "n_files_used": len(fractions),
            })
    return pd.DataFrame(rows)


feature_table = build_feature_comparison_table(results_folder, class_name_list, number_of_attention_branches)
timing_table = build_timing_table(dictionary, class_name_list, number_of_attention_branches)
print(f"feature_table: {feature_table.shape[0]} rows ({len(feature_table.to_csv(index=False).encode('utf-8'))} bytes as CSV)")
print(f"timing_table: {timing_table.shape[0]} rows")

# The key: the form field first, then the Colab secret ANTHROPIC_API_KEY.
if not anthropic_api_key.strip():
    try:
        from google.colab import userdata
        anthropic_api_key = userdata.get("ANTHROPIC_API_KEY") or ""
    except Exception:
        anthropic_api_key = ""

if anthropic_api_key.strip():
    try:
        import anthropic
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
        import anthropic

    prompt = f"""You are helping interpret an explainable-AI model (DiverScan) trained to tell apart two experimental conditions from animal trajectories, using {number_of_attention_branches} attention branches.

Experimental setting: {experimental_setting}
Conditions compared: {class_name_list[0]} vs {class_name_list[1]}

For each attention branch, read that branch's rows in both tables below and answer, in order:
1. Does the branch fire at noticeably different rates between the two conditions (`highlighted_timepoints_pct` in Table 2)? If yes, that is evidence the branch is tracking something that happens more often, or for longer, in one condition.
2. If the firing rate is similar between conditions, look instead at how the feature distributions differ *inside* the highlighted segments (Table 1's `mean_*` / `median_*` / `std_*` / `hellinger_distance` columns) — this tells you what differs about the behavior *during* the event the branch attends to, rather than how often it happens.
3. Use `highlighted_time_centroid_0to1` in Table 2 (0 = start of trajectory, 1 = end) and the correlation with `time` in Table 1 to say roughly *when* in the trajectory the branch's attention concentrates, and whether that differs between conditions.
4. Combine 1-3 with the experimental setting above to propose a concrete biological interpretation for that branch. Flag any branch whose top features look redundant with another branch's.

Be concise per branch (a short paragraph each, referencing concrete numbers from the tables). Do not run or suggest statistical significance tests. End with a 2-3 sentence overall summary across all branches.

Table 1 - per-branch, per-feature correlation / Hellinger distance / within-segment distribution summary (CSV):
{feature_table.to_csv(index=False)}

Table 2 - per-branch, per-condition firing rate and timing (CSV):
{timing_table.to_csv(index=False)}
"""

    client = anthropic.Anthropic(api_key=anthropic_api_key)
    response = client.messages.create(
        model=model_id,
        max_tokens=16000,
        messages=[{"role": "user", "content": prompt}],
    )
    if response.stop_reason == "refusal":
        print("The model declined to answer this request.")
        if response.stop_details:
            print("Reason:", response.stop_details.explanation)
    else:
        # The response holds content blocks; only the text blocks are the answer.
        answer = "".join(block.text for block in response.content if block.type == "text")
        display(Markdown(answer))
        if response.stop_reason == "max_tokens":
            print("\n(The answer was cut off at max_tokens; raise it to see the rest.)")

else:
    report_path = Path(results_folder) / "attention_report.html"
    if report_path.exists():
        print("No API key entered - showing the report prepared ahead of time instead.")
        display(HTML(filename=str(report_path)))
    else:
        print(
            "No API key entered, and no pre-made report found at "
            f"{report_path}. Either fill in `anthropic_api_key` above, or "
            "place an `attention_report.html` file next to the other result files."
        )
